# Data Analysis & Modeling: Which Submissions Should Enter Human Review?

**Original mock inspired by collected interview topics, not a reported Roblox coding question.** All data are synthetic. Format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. Explain your assumptions and choices while coding.

## Interviewer prompt

You work on a platform's content-review team. At submission time, a model should decide which submissions to send to a human reviewer. This is a queueing decision, not an automatic deletion or account ban.

**Problem 1 — Data analysis (14 minutes).** Load the submission and creator tables, inspect data quality, clean fixed validity errors, and build one row per submission. How common is the positive class, and does its observed rate vary by content type?

**Problem 2 — Modeling (21 minutes).** Train a simple binary classifier and a constant baseline. The team's practice requirement is to catch **at least 80% of abusive submissions on validation**, while making the review queue as small as possible among the model's available score thresholds. How would you select a threshold and evaluate it honestly? Report final test ranking metrics, precision, recall and the fraction of submissions entering review.

Explain your assumptions and choices out loud. Use a train/validation/test split; do not use test results to change the threshold. The 80% requirement is an exercise assumption, not a Roblox policy.

**What this mock exercises:** familiar cleaning, many-to-one joins, binary LogisticRegression and probability metrics; a separate validation set; **precision_recall_curve**; turning probabilities into a review decision.

This dataset is not assumed to have an extremely rare positive class: measure its actual prevalence. A useful classifier and a useful operating threshold are different things.

| Coding time | Focus |
| --- | --- |
| 0–4 min | Load the inputs and define a positive example |
| 4–8 min | Clean without deleting useful examples |
| 8–14 min | Join metadata and analyze the queue population |
| 14–18 min | Separate training, threshold selection and final evaluation |
| 18–25 min | Fit a classifier and a constant baseline |
| 25–31 min | Choose the review threshold on validation |
| 31–35 min | Evaluate the frozen policy and hand it off |

Suggested checkpoints only; they are not official grading criteria. The two problems are data analysis (Steps 1–3) and modeling (Steps 4–7).

**Practice version:** write your own imports, then complete each exercise and run its check cell. The questions are part of the mock; explain your answer before moving on. Run All intentionally stops at unfinished exercises.

## Data contract

The files are already supplied. Do not generate data or modify the source CSVs.

**submissions.csv — one logged submission per row before cleaning.**

| Column | Meaning |
| --- | --- |
| submission_id | Event identifier; identical exported rows can repeat |
| creator_id | Key into creators.csv |
| text_length | Number of characters; negative sentinel means unknown |
| link_count | Number of links |
| uppercase_ratio | Uppercase share, between 0 and 1 |
| repeated_token_ratio | Repetition share, between 0 and 1 |
| content_type | chat, description or comment; can be missing |
| label | Historical completed review: clean, spam or abuse |
| review_minutes | Time spent reviewing, available only after submission; not an input |

**creators.csv — one row per creator:** creator_id and account_age_days, measured at submission time; account age may be missing.

**Target for this exercise:** needs_review = 1 exactly when label is **abuse**, otherwise 0. Treating spam as negative is only a simplified abuse-specific queue definition, not a real moderation policy. Assume every historical row received a completed review, so the labels are available for training without selecting only previously flagged submissions.

All listed predictor fields are available at submission time. Exclude IDs, label, needs_review and review_minutes from model inputs. Each creator has one distinct submission in this fixture, so disjoint submission splits are also creator-disjoint. A future dataset with repeated creators would need a split matched to the deployment goal.

Negative text lengths become missing; keep their rows. Unknown predictors are not zero and do not make a valid label unusable. Fit imputers, scalers and category vocabularies on training data only.


## Setup

Use the workspace root as the working directory and select **.venv-codesignal/bin/python**. Inputs are in **data/codesignal_roblox_softmax/**. No downloads, data generation, GPU or new dependencies are needed.

Write the imports you need, define **SEED = 42**, and set **DATA_DIR** to the supplied directory. Include setup in your timed attempt. Restart the kernel first so earlier notebooks do not hide missing imports.

Do not edit the source CSVs. These exercises deliberately reuse existing files to focus on new decisions rather than a new dataset generator.


In [40]:
# Write your own imports, SEED and DATA_DIR.
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn import metrics

SEED = 42

#raise NotImplementedError("Complete setup before loading the CSVs.")


## Problem 1 · Step 1: Load the inputs and define a positive example (0–4 min)

Load **submissions** and **creators**. What does each row represent? Inspect shapes, types, missing values, label counts, exact duplicate rows and invalid text lengths. Confirm which fields are known when a new submission arrives.

**Say before coding:** "I will define the positive class and prediction time before choosing a model or dropping rows."

**Possible interviewer questions**

1. What is a positive example in this task?
2. Does having a label in the training file make it a valid feature?
3. What would change if only previously flagged submissions were reviewed?


In [3]:
# Implement the task above.
#
DATA_DIR = './data/codesignal_roblox_softmax/'

creators_dir = DATA_DIR + 'creators.csv'
submissions_dir = DATA_DIR + 'submissions.csv'

creators = pd.read_csv(creators_dir)
submissions = pd.read_csv(submissions_dir)

display(creators.head())
display(submissions.head())

print("duplicate:",  submissions.duplicated().sum() )
print("missing value:", submissions.isna().sum())

print("invalidate value:", submissions["text_length"].lt(0).sum())
print("invalidate value:", submissions["review_minutes"].lt(0).sum())




# #raise NotImplementedError("Complete step 1, then run its checks.")


,creator_id,account_age_days
0,0,1395.0
1,1,614.0
2,2,1387.0
3,3,124.0
4,4,480.0


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes
0,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918
1,1,1,505.0,2,0.223062,0.039823,comment,clean,1.170087
2,2,2,428.0,3,0.311449,0.349885,description,spam,5.320863
3,3,3,290.0,1,0.058440,0.084628,chat,spam,5.876839
4,4,4,287.0,1,0.479259,0.315910,description,clean,1.283389


duplicate: 15
missing value: submission_id            0
creator_id               0
text_length              0
link_count               0
uppercase_ratio          0
repeated_token_ratio     0
content_type            40
label                    0
review_minutes           0
dtype: int64
invalidate value: 35
invalidate value: 0


In [ ]:
assert creators["creator_id"].is_unique
assert submissions["label"].notna().all()
assert set(submissions["label"]) == {"clean", "spam", "abuse"}
assert submissions["creator_id"].isin(creators["creator_id"]).all()


## Problem 1 · Step 2: Clean without deleting useful examples (4–8 min)

Create **clean** without changing submissions. Remove identical exported rows. Check that submission_id is now unique; stop and investigate if it is not. Replace invalid negative text_length values with missing values, keeping those submissions. Do not learn imputation values yet.

**Say before coding:** "I will separate fixed validity rules from preprocessing statistics that must be fitted on training data."

**Possible interviewer questions**

1. Why not drop every row with a missing field?
2. Why is missing text length different from a length of zero?
3. Can removing exact duplicate rows guarantee unique event IDs?


In [ ]:
# Implement the task above.
cleaned_submissions = submissions.drop_duplicates().copy()

masked = submissions["text_length"].lt(0)
cleaned_submissions.loc[masked, "text_length"] = np.nan

num_submission_id = cleaned_submissions["submission_id"].nunique()
print("num_submission_id",num_submission_id)
print("len", len(cleaned_submissions))



##raise NotImplementedError("Complete step 2, then run its checks.")


num_submission_id 1200
len 1200


In [ ]:
assert clean["submission_id"].is_unique
assert clean["text_length"].dropna().ge(0).all()
assert len(clean) == len(submissions.drop_duplicates())
assert clean["text_length"].isna().sum() >= submissions["text_length"].lt(0).sum() - submissions.duplicated().sum()


## Problem 1 · Step 3: Join metadata and analyze the queue population (8–14 min)

Create **frame** with one row per submission and creator metadata attached. Define **needs_review** from the historical label. Report its prevalence overall and a **content_summary** with submission counts and abuse rates by content_type, retaining the missing-type group. Explain what these rates can and cannot establish.

**Say before coding:** "I will preserve the submission unit and check that the metadata join does not multiply examples."

**Possible interviewer questions**

1. Why validate this join as many-to-one?
2. Does a higher abuse rate for one content type prove that the type causes abuse?
3. Would accuracy alone tell us whether this review queue is useful?


In [20]:
# Implement the task above.
frame = cleaned_submissions.merge(creators, on='creator_id', how='left', validate="many_to_one")

#display(frame.head())

needs_review = frame["label"].eq("abuse")
frame["needs_review"] = frame["label"].eq("abuse")

display(frame.head())
display(frame.loc[needs_review].head())
content_summary = frame.groupby("content_type").agg(
    submissions = ("submission_id", "size"),
    absue_rates = ("needs_review", "mean"),
)



#raise NotImplementedError("Complete step 3, then run its checks.")


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes,account_age_days,needs_review
0,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918,1395.0,True
1,1,1,505.0,2,0.223062,0.039823,comment,clean,1.170087,614.0,False
2,2,2,428.0,3,0.311449,0.349885,description,spam,5.320863,1387.0,False
3,3,3,290.0,1,0.058440,0.084628,chat,spam,5.876839,124.0,False
4,4,4,287.0,1,0.479259,0.315910,description,clean,1.283389,480.0,False


,submission_id,creator_id,text_length,link_count,uppercase_ratio,repeated_token_ratio,content_type,label,review_minutes,account_age_days,needs_review
0,0,0,67.0,0,0.254411,0.048013,description,abuse,5.368918,1395.0,True
15,15,15,513.0,0,0.191396,0.495664,comment,abuse,5.741762,1119.0,True
17,17,17,91.0,0,0.200471,0.098816,description,abuse,5.432234,NaN,True
24,24,24,510.0,0,0.349163,0.205014,chat,abuse,5.605020,1173.0,True
34,34,34,578.0,3,0.374773,0.262218,comment,abuse,5.848034,1422.0,True


In [ ]:
assert len(frame) == len(clean)
assert frame["submission_id"].is_unique
np.testing.assert_array_equal(frame["needs_review"], frame["label"].eq("abuse").astype(int))
assert content_summary["submissions"].sum() == len(frame)


## Problem 2 · Step 4: Separate training, threshold selection and final evaluation (14–18 min)

Which columns should be features, and which should be excluded? Create **numeric_features**, **categorical_features**, **X** and **y**. Create disjoint **X_train / y_train**, **X_valid / y_valid** and **X_test / y_test**, approximately 60% / 20% / 20%, preserving class balance. Explain the role of each set.

**Say before coding:** "Training fits the model, validation chooses the threshold, and the untouched test set estimates the final policy."

**Possible interviewer questions**

1. Why does this task need validation as well as test?
2. Why stratify these splits?
3. What if the deployment goal were performance on next month's submissions?


In [ ]:
# Implement the task above.
numeric_features = ["text_length","link_count","uppercase_ratio", "repeated_token_ratio","account_age_days"]
categorical_features = ["content_type"]

X = frame[numeric_features + categorical_features]
y = frame["needs_review"]

### train 0.6, val 0.2 test 0.2

### dev 0.8 test 0.2 ##

X_dev, X_test, y_dev, y_test = train_test_split(X,y,test_size=0.2,random_state=SEED)

X_train, X_val, y_train, y_val = train_test_split(X_dev,y_dev, test_size=0.25,random_state=SEED)

#check the ratio
##
print(y_dev.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))



#raise NotImplementedError("Complete step 4, then run its checks.")


needs_review
False    0.760417
True     0.239583
Name: proportion, dtype: float64
needs_review
False    0.770833
True     0.229167
Name: proportion, dtype: float64


In [ ]:
for left, right in [(X_train, X_valid), (X_train, X_test), (X_valid, X_test)]:
    assert set(left.index).isdisjoint(right.index)
for inputs, target in [(X_train, y_train), (X_valid, y_valid), (X_test, y_test)]:
    assert inputs.index.equals(target.index)
    assert set(target) == {0, 1}
assert len(X_train) + len(X_valid) + len(X_test) == len(frame)
assert not {"label", "needs_review", "review_minutes", "creator_id", "submission_id"} & set(X.columns)


## Problem 2 · Step 5: Fit a classifier and a constant baseline (18–25 min)

Fit **model**, a LogisticRegression pipeline with numeric imputation/scaling and categorical imputation/encoding. Fit **baseline**, a DummyClassifier using training class frequencies. Use training data only. What objective is the classifier minimizing, and why is the constant model a useful comparison?

**Say before coding:** "I will start with a small interpretable model and keep every learned preprocessing statistic inside its pipeline."

**Possible interviewer questions**

1. What does LogisticRegression learn?
2. Does class_weight="balanced" remove the need to choose a threshold?
3. What does the prior baseline predict?


In [ ]:
# Implement the task above.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("normalizer", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encoder", OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preproceesor", preprocess),
    ("classifier", LogisticRegression()),
])

baseline_classifier = DummyClassifier(strategy='prior')

baseline_classifier.fit(X_train, y_train)
model.fit(X_train,y_train)


## cross entroy. loss = plogp + (1-p)log1-p



#raise NotImplementedError("Complete step 5, then run its checks.")


Pipeline(steps=[('preproceesor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('normalizer',
                                                                   StandardScaler())]),
                                                  ['text_length', 'link_count',
                                                   'uppercase_ratio',
                                                   'repeated_token_ratio',
                                                   'account_age_days']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder())]),
                                                  ['content_type'])])),
                ('classifier', LogisticRegression())])

In [ ]:
np.testing.assert_array_equal(model.classes_, [0, 1])
learned_medians = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned_medians, X_train[numeric_features].median().to_numpy())
np.testing.assert_allclose(baseline.class_prior_[1], y_train.mean())


## Problem 2 · Step 6: Choose the review threshold on validation (25–31 min)

Create **valid_prob**. Build a **threshold_table** describing precision and recall at the available validation score thresholds. Choose **threshold** to meet validation recall of at least **recall_target = 0.80**, while sending as few validation submissions as possible for review.

What does lowering the threshold do to queue volume? Report **validation_decision**, the selected precision, recall and review fraction. Use validation only. If the requirement cannot be evaluated or no candidate is feasible, explain that rather than inventing a successful result.

**Say before coding:** "I am choosing an operating point for a stated review objective; this is not a promise that future recall will equal validation recall."

**Possible interviewer questions**

1. How does precision_recall_curve align its outputs?
2. Why choose the largest threshold that still meets the recall requirement?
3. Would you use the same threshold for automatic deletion?


In [45]:
# Implement the task above.
thresholds = [0.1,0.15,0.2,0.3,0.4,0.5,0.6,0.7]

print(model.classes_)
valid_prob = model.predict_proba(X_val)[:,1]
#display(valid_prob)


def evaluate(gt, prediction, threshold):
    pred = (prediction > threshold).astype(int)

    return {
        "threshold": threshold,
        "recall": metrics.recall_score(gt,pred),
    }

recall_scores = []

for threshold in thresholds:
    recall_scores.append(evaluate(y_val,valid_prob,threshold))


recall_table = pd.DataFrame(recall_scores)

display(recall_table)

    






#raise NotImplementedError("Complete step 6, then run its checks.")


[False  True]


,threshold,recall
0,0.10,0.985075
1,0.15,0.880597
2,0.20,0.776119
3,0.30,0.552239
4,0.40,0.447761
5,0.50,0.268657
6,0.60,0.179104
7,0.70,0.104478


In [ ]:
assert len(threshold_table) == len(thresholds)
assert len(precision) == len(recall) == len(thresholds) + 1
assert metrics.recall_score(y_valid, validation_decision) >= recall_target
assert threshold == feasible["threshold"].max()
assert ((validation_decision == 0) | (validation_decision == 1)).all()


## Problem 2 · Step 7: Evaluate the frozen policy and hand it off (31–35 min)

Freeze the fitted model and selected threshold. Create **test_prob**, **test_decision** and **scores** comparing the prior baseline and model on ROC-AUC, AP and log loss. Create **policy_scores** comparing threshold 0.5 and the selected threshold on precision, recall, F1 and review fraction.

Does the validation recall target also hold on test? Report the result honestly without adjusting the policy. Summarize the trade-off and what you would monitor after deployment. Confusion-matrix display and new-submission inference are optional handoff follow-ups below.

**Say before coding:** "The test set evaluates one frozen policy. I will report both abuse coverage and the human-review workload, including any missed target."

**Possible interviewer questions**

1. Why do ROC-AUC and AP receive probabilities rather than 0/1 decisions?
2. Why might test recall fall below 80%?
3. What if there are not enough reviewers for the resulting queue?


In [49]:
# Implement the task above.
threshold = 0.15

test_prob = model.predict_proba(X_test)[:,1]
test_decision = (test_prob > threshold).astype(int)

display(test_decision)


def ranking_scores(gt, predictions):
    return {
        "ROC_AUC": metrics.roc_auc_score(gt,predictions),
        "AP": metrics.average_precision_score(gt,predictions),
        "log_loss": metrics.log_loss(gt,predictions),
    }


summarized_scores = pd.DataFrame(
    {
        "prior": ranking_scores(y_test,baseline_classifier.predict_proba(X_test)[:,1]),
        "logstic": ranking_scores(y_test,test_prob),
    }
)

display(summarized_scores)



#raise NotImplementedError("Complete step 7, then run its checks.")


array([1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1,
       0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1,
       1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0,
       1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0,
       1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1])

,prior,logstic
ROC_AUC,0.500000,0.720688
AP,0.229167,0.406688
log_loss,0.538289,0.495668


In [ ]:
assert test_prob.shape == test_decision.shape == (len(y_test),)
assert np.isfinite(test_prob).all()
np.testing.assert_allclose(scores.loc["prior", "AP"], y_test.mean())
np.testing.assert_allclose(scores.loc["prior", "ROC_AUC"], 0.5)
np.testing.assert_allclose(policy_scores.loc["selected_threshold", "review_fraction"], test_decision.mean())


## Optional follow-ups — outside the 35-minute core

**Handoff follow-up:** Show a confusion matrix. Score one new_submission with a missing numeric field and an unseen content_type, creating new_probability and send_to_review with the frozen model and threshold. Do not fit again.

**Capacity follow-up:** A reviewer can inspect only 10% of submissions. Is that compatible with your recall goal? Use validation to inspect both constraints before touching test. Discuss why a fixed score threshold does not guarantee a fixed review count after drift.

**New-data follow-up:** The abuse fraction changes next month. What would you monitor, and why might precision move even if the model's conditional detection rates did not change?

**API notes, if needed:** precision_recall_curve produces arrays across score thresholds; unlike ROC-AUC/AP, precision_score and recall_score expect the chosen class decisions.


## Scope and sources

The user-supplied **Data Analysis & Modeling Coding Interview** guide supplies the format: data loading, analysis, feature engineering, a baseline model, evaluation and live explanation in a CodeSignal notebook. It does not specify these exact tasks or APIs.

The local interview collection separates reported **Question/Asked** topics from our explanatory **Answer** material. This mock adapts the topics for practice; it is not evidence that Roblox used this coding problem. Synthetic rates and thresholds are not real product statistics or policy.

The collected report topics motivating this original mock are harmful-content detection ([1125288](https://www.1point3acres.com/bbs/thread-1125288-1-1.html)), classification loss ([1147380](https://www.1point3acres.com/bbs/thread-1147380-1-1.html)), and evaluation ([1161498](https://www.1point3acres.com/bbs/thread-1161498-1-1.html)). The source index is [08_company_interviews.md](08_company_interviews.md). Those reports do **not** establish this exact notebook task, the 80% requirement or these CSVs as actual interview material.

Implementation reference: [sklearn precision_recall_curve](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.precision_recall_curve.html).
